# Notebook 03_04 — Feature Engineering: AutoEncoder

Shallow neural autoencoder: `D -> K (bottleneck, ReLU) -> D` trained to reconstruct its
input (unsupervised). The encoder half (`D -> K`) is then used as the feature transform.

**Why subsample for fitting**: training `MLPRegressor` directly on 3.4M rows for up to
200 epochs is extremely slow with sklearn's solver. We fit on a **stratified subsample**
(`AE_FIT_SAMPLE_SIZE = 200,000`) — the encode step afterwards is a cheap matrix multiply
(`ReLU(X·W + b)`) applied to the FULL train/val/test sets, so this does not bias which
samples get transformed, only which samples shape the encoder's weights.

**Results saved incrementally to** `results/fe_autoencoder_raw.csv` — resumable on crash.

In [ ]:
import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CODE_DIR = '/content/UAV' if IN_COLAB else os.environ.get('UAV_CODE_DIR', 'D:/UAV_')

if IN_COLAB and not os.path.isdir(os.path.join(CODE_DIR, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/haribawngg/UAV.git', CODE_DIR], check=True)

sys.path.insert(0, os.path.join(CODE_DIR, 'notebook'))
from common import *
from sklearn.neural_network import MLPRegressor

print('Imports OK')

In [ ]:
d = load_data()
X_train, X_val, X_test = d['X_train'], d['X_val'], d['X_test']
y_train = d['y_train']

METHOD_NAME = 'AutoEncoder'
RESULTS_CSV = f'{RESULTS_DIR}fe_autoencoder_raw.csv'

AE_FIT_SAMPLE_SIZE = 200_000

In [ ]:
def transform_fn(K):
    X_fit, _ = stratified_subsample(X_train, y_train, AE_FIT_SAMPLE_SIZE, seed=42)

    t0 = time.time()
    ae = MLPRegressor(
        hidden_layer_sizes=(K,),   # bottleneck = K
        activation='relu',
        max_iter=200,
        random_state=42,
        early_stopping=True
    )
    ae.fit(X_fit, X_fit)  # reconstruct input (unsupervised)
    print(f'  [AutoEncoder K={K}] fit on {len(X_fit):,} samples in {time.time()-t0:.1f}s  '
          f'(n_iter={ae.n_iter_})')

    W, b = ae.coefs_[0], ae.intercepts_[0]

    def encode(X):
        return np.maximum(0, X @ W + b)  # ReLU(X.W + b) -- the encoder half only

    return encode(X_train), encode(X_val), encode(X_test), K

In [ ]:
run_experiment_grid(
    method_name=METHOD_NAME,
    transform_fn=transform_fn,
    d=d,
    results_csv_path=RESULTS_CSV
)

In [ ]:
res_df = pd.read_csv(RESULTS_CSV)
print(res_df.groupby(['K', 'classifier'])[['f1', 'train_time_s', 'inference_ms']].agg(['mean', 'std']).to_string())